# Lags y Ventanas Móviles
## El alma del feature engineering temporal

> **Fase 2 — Video 9** | Feature Engineering
> Dataset: Histórico de Ventas Sintético

---

### Convertir la *memoria* de la serie en columnas

En el Video 5 vimos que la serie **tiene memoria**: la demanda de hoy se parece a la de ayer, a la de hace
una semana y a la del mismo período del año pasado (ACF). Los **lags** y las **ventanas móviles** son la
forma de meter esa memoria en un dataset tabular para que un modelo de ML (XGBoost, del Video 15) la
aproveche. Son, literalmente, el alma del feature engineering temporal.

Pero traen consigo el error más peligroso —y más común— de todo el forecasting: la **fuga de datos** (*data
leakage*). Un feature que sin querer mira el futuro te dará métricas espectaculares… que se derrumban en
producción. Este video enseña a construir estos features **y** a no envenenarlos.

### Ruta del notebook
1. Librerías y carga
2. Lags: la memoria directa
3. Ventanas móviles (rolling): tendencia y volatilidad local
4. Expanding y EWM: historia acumulada y memoria con olvido
5. ⚠️ El error mortal: fuga de datos con ventanas centradas
6. El patrón correcto: features sin fuga, por SKU
7. Conclusiones y puente al Video 10

---
## 1. Librerías y carga de datos

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from pathlib import Path

plt.rcParams.update({
    'figure.facecolor': '#F8FAFC', 'axes.facecolor': '#F8FAFC',
    'axes.grid': True, 'grid.color': '#E2E8F0',
    'grid.linewidth': 0.8, 'font.size': 11,
})
BLUE, RED, GREEN, PURPLE, ORANGE = '#2563EB','#DC2626','#16A34A','#7C3AED','#EA580C'
unit_fmt = FuncFormatter(lambda x, _: f'{x/1e3:.0f}k')
print('✅ Librerías cargadas')

In [ ]:
def find_csv(filename='sales_history.csv', max_levels=4):
    base = Path().resolve()
    for _ in range(max_levels):
        candidate = base / 'output' / filename
        if candidate.exists():
            return candidate
        base = base.parent
    raise FileNotFoundError(f"No se encontró '{filename}'. Corre main.py primero.")

csv_path = find_csv()
df = pd.read_csv(csv_path, parse_dates=['date']).sort_values('date')

# Serie semanal de demanda total (quitamos las semanas parciales de los bordes)
y = (df.groupby(pd.Grouper(key='date', freq='W-MON'))['units_sold'].sum()
       .iloc[1:-1])
y.name = 'demanda'

print(f'✅ Serie semanal: {len(y)} semanas | {y.index.min().date()} → {y.index.max().date()}')

---
## 2. Lags: la memoria directa

Un **lag** es simplemente la serie desplazada *k* pasos: `y.shift(k)`. El feature `lag_k` en la fila `t`
contiene el valor de hace `k` semanas — información que **sí** está disponible al momento de pronosticar.

¿Qué lags elegir? Los que la **ACF/PACF** (Video 5) marcaron como significativos. Para datos semanales con
ciclo anual, esperamos que `lag_52` (el mismo período del año pasado) sea oro puro.

In [ ]:
lags_to_test = [1, 2, 3, 4, 8, 12, 26, 52]
corrs = {k: y.shift(k).corr(y) for k in lags_to_test}

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
# Panel 1: la serie con un par de lags
axes[0].plot(y.index, y.values, color=BLUE, linewidth=1.5, label='demanda (t)')
axes[0].plot(y.index, y.shift(1).values, color=ORANGE, linewidth=1, alpha=0.8, label='lag 1')
axes[0].plot(y.index, y.shift(52).values, color=GREEN, linewidth=1, alpha=0.8, label='lag 52')
axes[0].yaxis.set_major_formatter(unit_fmt)
axes[0].set_title('La serie y dos de sus lags', fontsize=12, fontweight='bold')
axes[0].legend(loc='upper left')

# Panel 2: correlación de cada lag con el presente
bars = axes[1].bar([str(k) for k in lags_to_test], [corrs[k] for k in lags_to_test],
                   color=[GREEN if k == 52 else BLUE for k in lags_to_test], edgecolor='white')
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_title('Correlación de cada lag con la demanda actual', fontsize=12, fontweight='bold')
axes[1].set_xlabel('lag (semanas)')

fig.suptitle('Lags: convertir la memoria en columnas', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('Correlación lag ↔ demanda actual:')
for k in lags_to_test:
    star = '  ← el mismo período del año pasado' if k == 52 else ''
    print(f'  lag {k:>2}: {corrs[k]:+.3f}{star}')

---
## 3. Ventanas móviles (rolling): tendencia y volatilidad local

Un lag da un punto; una **ventana móvil** resume los últimos *w* puntos: media (nivel reciente), desviación
(volatilidad), min/max (rango). Es un suavizado que le da al modelo el "clima" reciente de la serie.

> 🔑 **La regla de oro:** para pronosticar `t` solo puedes usar información **hasta `t-1`**. Por eso
> siempre `shift(1)` **antes** de la ventana. `y.shift(1).rolling(w)` mira `[t-w, …, t-1]`; nunca incluye
> el presente. Omitir ese `shift(1)` es la primera forma de fuga de datos.

In [ ]:
W = 8
base = y.shift(1)                       # <- clave: desplazar antes de la ventana
roll_mean = base.rolling(W).mean()
roll_std  = base.rolling(W).std()
roll_min  = base.rolling(W).min()
roll_max  = base.rolling(W).max()

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
axes[0].plot(y.index, y.values, color=BLUE, linewidth=1, alpha=0.5, label='demanda')
axes[0].plot(roll_mean.index, roll_mean.values, color=RED, linewidth=2, label=f'media móvil {W} (trailing)')
axes[0].fill_between(roll_min.index, roll_min.values, roll_max.values, color=ORANGE, alpha=0.15,
                     label='rango min–max')
axes[0].yaxis.set_major_formatter(unit_fmt)
axes[0].set_title(f'Media móvil y rango (ventana={W}, con shift(1))', fontsize=12, fontweight='bold')
axes[0].legend(loc='upper left')

axes[1].plot(roll_std.index, roll_std.values, color=PURPLE, linewidth=1.8)
axes[1].yaxis.set_major_formatter(unit_fmt)
axes[1].set_title('Desviación estándar móvil (volatilidad reciente)', fontsize=12, fontweight='bold')

fig.suptitle('Ventanas móviles: el clima reciente de la serie', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Features rolling (ventana {W}) correlación con la demanda actual:')
for name, s in [('rollmean', roll_mean), ('rollstd', roll_std), ('rollmax', roll_max)]:
    print(f'  {name:<9}: {s.corr(y):+.3f}')

---
## 4. Expanding y EWM: historia acumulada y memoria con olvido

- **Expanding:** una ventana que crece — usa *toda* la historia hasta `t-1`. Útil como "nivel base
  histórico" del SKU. `y.shift(1).expanding().mean()`.
- **EWM** (media móvil exponencial): pondera más lo reciente y olvida el pasado de forma gradual. El
  parámetro `span` controla cuánta memoria: span chico = reacciona rápido; span grande = más suave.

In [ ]:
exp_mean  = y.shift(1).expanding(min_periods=4).mean()
ewm_fast  = y.shift(1).ewm(span=4).mean()
ewm_slow  = y.shift(1).ewm(span=26).mean()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(y.index, y.values, color=BLUE, linewidth=1, alpha=0.4, label='demanda')
ax.plot(exp_mean.index, exp_mean.values, color=GREEN, linewidth=2, label='expanding mean (toda la historia)')
ax.plot(ewm_fast.index, ewm_fast.values, color=RED, linewidth=1.8, label='EWM span=4 (reacciona rápido)')
ax.plot(ewm_slow.index, ewm_slow.values, color=PURPLE, linewidth=1.8, label='EWM span=26 (suave)')
ax.yaxis.set_major_formatter(unit_fmt)
ax.set_title('Expanding vs EWM', fontsize=13, fontweight='bold')
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

print('El expanding mean es estable pero lento para reaccionar a cambios de nivel;')
print('el EWM span=4 sigue de cerca la serie; el span=26 la suaviza. Todos usan shift(1).')

---
## 5. ⚠️ El error mortal: fuga de datos (*data leakage*)

Aquí es donde mueren los proyectos de forecasting. Una **ventana centrada** —`rolling(w, center=True)`—
promedia puntos a ambos lados de `t`: incluye el **presente y el futuro**. Como feature para predecir `t`,
es hacer trampa: estás usando la respuesta.

El síntoma es delicioso y traicionero: **las métricas mejoran muchísimo**. Vamos a demostrarlo comparando
un feature válido (trailing, con `shift(1)`) contra uno con fuga (centrado), entrenando el mismo modelo.

In [ ]:
valid_feature  = y.shift(1).rolling(4).mean()      # ✅ solo pasado
leaked_feature = y.rolling(4, center=True).mean()   # ❌ incluye presente y futuro

print('Correlación con la demanda actual:')
print(f'  ✅ trailing (shift1, w=4): {valid_feature.corr(y):+.3f}')
print(f'  ❌ centrada  (center, w=4): {leaked_feature.corr(y):+.3f}   ← sospechosamente alta\n')

def fit_eval(feature):
    d = pd.concat([y.rename('y'), feature.rename('f')], axis=1).dropna()
    n = int(len(d) * 0.7)
    tr, te = d.iloc[:n], d.iloc[n:]
    m = LinearRegression().fit(tr[['f']], tr['y'])
    return r2_score(tr['y'], m.predict(tr[['f']])), r2_score(te['y'], m.predict(te[['f']]))

r2_valid_tr, r2_valid_te   = fit_eval(valid_feature)
r2_leak_tr,  r2_leak_te    = fit_eval(leaked_feature)

fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(2)
ax.bar(x - 0.2, [r2_valid_tr, r2_leak_tr], 0.4, label='train', color=BLUE)
ax.bar(x + 0.2, [r2_valid_te, r2_leak_te], 0.4, label='test',  color=ORANGE)
ax.set_xticks(x); ax.set_xticklabels(['✅ Válido (trailing)', '❌ Con fuga (centrado)'])
ax.set_ylabel('R²'); ax.set_title('El espejismo de la fuga de datos', fontsize=13, fontweight='bold')
ax.legend()
for i, v in enumerate([r2_valid_te, r2_leak_te]):
    ax.text(i + 0.2, v + 0.01, f'{v:.2f}', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

print(f'R² test  — válido: {r2_valid_te:.3f}   con fuga: {r2_leak_te:.3f}')
print(f'La fuga infla el R² de test ×{r2_leak_te / r2_valid_te:.1f}. Parece un modelo mucho mejor…')
print('\n…pero es una mentira. Y la prueba definitiva:')
print(f'  Último valor de la feature CENTRADA: {leaked_feature.iloc[-1]}')
print('  → es NaN: para calcularla en la última semana necesitas semanas FUTURAS que no existen.')
print('  En producción, ese feature es imposible de generar. El "mejor" modelo no se puede usar.')

---
## 6. El patrón correcto: features sin fuga, por SKU

En la práctica los features se construyen **por SKU** (cada serie con su propia historia) y siempre con
`shift` antes de cualquier agregación. Usar `groupby(sku)` evita además otra fuga sutil: mezclar la
historia de un producto con la de otro. Encapsulamos el patrón en una función reutilizable.

In [ ]:
def make_temporal_features(frame, key='sku_id', time='date', target='units_sold',
                           lags=(1, 2, 4, 52), windows=(4, 12)):
    # Lags + ventanas móviles SIN fuga de datos, calculados por SKU
    f = frame.sort_values([key, time]).copy()
    grp = f.groupby(key)[target]
    for L in lags:
        f[f'lag_{L}'] = grp.shift(L)
    for w in windows:
        # shift(1) ANTES del rolling → la ventana solo ve el pasado
        f[f'rollmean_{w}'] = grp.transform(lambda s: s.shift(1).rolling(w).mean())
        f[f'rollstd_{w}']  = grp.transform(lambda s: s.shift(1).rolling(w).std())
    f['ewm_8'] = grp.transform(lambda s: s.shift(1).ewm(span=8).mean())
    return f

# Demostración sobre demanda semanal por SKU
weekly_sku = (
    df.groupby(['sku_id', pd.Grouper(key='date', freq='W-MON')])['units_sold']
      .sum().reset_index()
)
feat = make_temporal_features(weekly_sku)

sku_demo = df.groupby('sku_id')['revenue'].sum().idxmax()
print(f'Features temporales para {sku_demo} (primeras filas con historia):')
cols = ['date', 'units_sold', 'lag_1', 'lag_52', 'rollmean_4', 'rollstd_4', 'ewm_8']
print(feat[feat['sku_id'] == sku_demo][cols].dropna().head(6).to_string(index=False))

print(f'\nMatriz final: {feat.shape[0]} filas × {feat.shape[1]} columnas '
      f'({feat.shape[1] - 3} features temporales)')
print('Todas las features usan solo el pasado → listas para el modelo del Video 15 sin fuga.')

---
## 7. Conclusiones y puente al Video 10

### Las reglas que te llevas

1. **Lags = memoria directa.** Elige los lags que la ACF/PACF (Video 5) señaló; en datos semanales con
   ciclo anual, `lag_52` suele ser el más informativo.
2. **Ventanas móviles = clima reciente:** media (nivel), std (volatilidad), min/max (rango).
3. **`shift(1)` SIEMPRE antes de la ventana.** Para predecir `t` solo existe información hasta `t-1`.
4. **EWM cuando la recencia importa;** expanding para un nivel base histórico.
5. **La fuga de datos es el enemigo #1.** Ventanas centradas (o cualquier feature que mire el futuro)
   inflan las métricas y son **imposibles de calcular en producción**. Si un resultado se ve demasiado
   bueno, sospecha de fuga antes que celebrar.
6. **Construye por SKU** (`groupby`) para no mezclar historias ni fugar entre series.

### El mapa

Ya tienes las dos familias de features: **calendario** (Video 8, exógenas conocidas del futuro) y
**memoria** (lags/ventanas, este video). Falta la palanca de negocio más poderosa del retail: **el precio
y la promoción**.

---

### Próximo video
**Video 10 — Precio y promoción: elasticidad, codificación y el impacto en pesos**
Mediremos la elasticidad precio de la demanda, codificaremos las promociones y traduciremos un error de
modelado a dinero: cuánto cuesta en sobre-inventario subestimar el efecto promocional.